# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")
print(f"Identifier: {metadata['identifier']}")
print(f"Keywords: {metadata['keywords']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate record sets, then show fields for each record set. All references use the `@id`.

In [ ]:
# List all available record sets by @id and name
record_sets = dataset.metadata.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# Show fields for each record set, referenced by @id
for rs in record_sets:
    print(f"\nFields in Record Set '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field.id})")

## 2.1 Show some sample records
Print a few records for a selected record set. We'll demonstrate with the first listed record set.

In [ ]:
# Print a few sample records for a record set
if record_sets:
    first_record_set_id = record_sets[0].id
    print(f"Sample records for Record Set '{record_sets[0].name}' (@id: {first_record_set_id}):")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis.

All references use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set

# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns for the first loaded record set
print(f"Columns for '{record_sets[0].name}' (@id: {record_sets[0].id}):")
print(dataframes[record_sets[0].id].columns.tolist())
dataframes[record_sets[0].id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a numeric field (e.g. GAD-7 score) from the main record set for analysis. All entities are referenced via their `@id`.

In [ ]:
# EDA: Filter, normalize, and group data

# Assume the main record set is the first
main_record_set = record_sets[0]
main_record_set_id = main_record_set.id
df = dataframes[main_record_set_id]

# Find a numeric field (e.g., GAD-7 score) via @id
numeric_field_id = None
for field in main_record_set.fields:
    if field.data_type in ['Float', 'Integer', 'Number'] and 'gad' in field.name.lower():
        numeric_field_id = field.id
        numeric_field_name = field.name
        break
if numeric_field_id is None:
    # Fallback: first numeric field
    for field in main_record_set.fields:
        if field.data_type in ['Float', 'Integer', 'Number']:
            numeric_field_id = field.id
            numeric_field_name = field.name
            break

# Choose a threshold for filtering
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())
    
    # Normalize
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])
    
    # Group by a categorical field, e.g. gender or education
    group_field_id = None
    preferred_groups = ['gender', 'education', 'village']
    for field in main_record_set.fields:
        if any(pg in field.name.lower() for pg in preferred_groups):
            group_field_id = field.id
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        print(grouped_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will visualize the normalized numeric field distribution and its relationship with a categorical grouping.

In [ ]:
# Visualization section
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of '{numeric_field_id}' ({numeric_field_name})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Compare groups if group_field_id is available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.suptitle("")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed survey records on mental health, demographics, and assessment scores.
- Numeric analysis (e.g. GAD-7 score) reveals the distribution and potentially high scores among select participants.
- Grouped analysis shows variation by demographic factors (e.g. gender, education).
- The dataset, referenced by Croissant schema `@id` fields, can be efficiently loaded and analyzed with `mlcroissant`.
- Further research and modeling can be conducted based on these findings, e.g. to predict risk or inform health policies.